In [ ]:
from pathlib import Path
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
import torch
import torch.nn as nn
import mne
import mne
from mne.preprocessing import ICA
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from moabb.datasets.bids_interface import StepType
import math

import mlflow

In [ ]:
cwd = Path.cwd()
dataset_folder = cwd.parent.parent / "datasets"
BCICIV_2A = dataset_folder / "bciciv_2a"
model_folder = cwd / "model"
model_folder.mkdir(exist_ok=True)
BCICIV_2A.mkdir(exist_ok=True)

assert dataset_folder.exists(), f"Dataset folder does not exist: {dataset_folder}"
assert BCICIV_2A.exists(), f"BCICIV folder does not exist: {BCICIV_2A}"
assert model_folder.exists(), f"Model folder does not exist: {model_folder}"

print(f"Dataset folder: {dataset_folder}")
print(f"BCICIV folder: {BCICIV_2A}")
print(f"Model folder: {model_folder}")

Dataset folder: /Users/ratchanonkhongsawi/Desktop/CMKL/3rd/S2/Research/gans_eeg/gans_eeg/datasets
BCICIV folder: /Users/ratchanonkhongsawi/Desktop/CMKL/3rd/S2/Research/gans_eeg/gans_eeg/datasets/bciciv_2a
Model folder: /Users/ratchanonkhongsawi/Desktop/CMKL/3rd/S2/Research/gans_eeg/gans_eeg/pluem/unet-gans/model


In [ ]:
mne.set_config("MNE_DATA", str(BCICIV_2A), set_env=True)
mne.set_log_level("CRITICAL")  # Suppress verbose MNE logging
print("MNE_DATA =", mne.get_config("MNE_DATA"))

MNE_DATA = /Users/ratchanonkhongsawi/Desktop/CMKL/3rd/S2/Research/gans_eeg/gans_eeg/datasets/bciciv_2a


In [ ]:
def apply_notch(raw, freqs=50.0):
    raw = raw.copy().load_data()
    raw.notch_filter(freqs=freqs, picks="eeg")
    return raw

In [ ]:
def remove_eog(raw):
    raw = raw.copy().load_data()
    raw.pick("eeg")
    for ch in raw.ch_names:
        print(ch)
    return raw

In [ ]:
def butterworth_IIR_filter(raw, low_freq=8, high_freq=30, order=4):
    raw = raw.copy().load_data()

    iir_params = dict(
        order=order,
        ftype='butter',
        output='sos'
    )

    raw.filter(
        l_freq=low_freq,
        h_freq=high_freq,
        method='iir',
        iir_params=iir_params,
        picks="eeg"
    )

    return raw

In [ ]:
def sliding_windows(X, window_size=50, step_size=50):
    """
    X: (n_epochs, n_channels, n_times)
    """
    n_epochs, n_channels, n_times = X.shape

    windows = []

    for ep in X:
        for start in range(0, n_times - window_size + 1, step_size):
            end = start + window_size
            windows.append(ep[:, start:end])

    return np.stack(windows, axis=0)

In [ ]:
def minmax_scale(X, eps=1e-12):
    """
    X: (n_windows, n_channels, n_times)
    Returns: scaled to [-1, 1]
    """
    X_min = X.min(axis=2, keepdims=True)
    X_max = X.max(axis=2, keepdims=True)

    X = (X - X_min) / (X_max - X_min + eps)  # [0,1]
    X = X * 2 - 1  # [-1,1]

    return X

In [ ]:
dataset = BNCI2014_001()
print(dataset.data_path(subject=1, path=BCICIV_2A, force_update=False, update_path=True, verbose=False))  # Download the data for subject 1 if not already downloaded
paradigm = MotorImagery(n_classes=1, tmin=0.0, tmax=3.0 - 1/250, events=['left_hand']) # 3s trial duration per paper, excludes last sample

['/Users/ratchanonkhongsawi/Desktop/CMKL/3rd/S2/Research/gans_eeg/gans_eeg/datasets/bciciv_2a/MNE-bnci-data/~bci/database/001-2014/A01T.mat', '/Users/ratchanonkhongsawi/Desktop/CMKL/3rd/S2/Research/gans_eeg/gans_eeg/datasets/bciciv_2a/MNE-bnci-data/~bci/database/001-2014/A01E.mat']


In [ ]:
process_pipelines = paradigm.make_process_pipelines(dataset)
for process_pipeline in process_pipelines:
    process_pipeline.remove_step(index=1)  # Remove the original notch filter step
    process_pipeline.insert_step(StepType.RAW, FunctionTransformer(apply_notch, validate=False, kw_args={"freqs": 50.0}), index=0)  # Insert the new notch filter step at the beginning
    process_pipeline.insert_step(StepType.RAW, FunctionTransformer(butterworth_IIR_filter, validate=False, kw_args={"low_freq": 4, "high_freq": 40, "order": 3}), index=2)  # Insert the 3rd order bandpass filter after the remove_eog step
    process_pipeline.insert_step(StepType.RAW, FunctionTransformer(butterworth_IIR_filter, validate=False, kw_args={"low_freq": 4, "high_freq": 40, "order": 5}), index=3)  # Insert the 5th order bandpass filter after the 3rd order bandpass filter



In [ ]:
postprocess_pipeline = Pipeline([
    ("sliding_windows", FunctionTransformer(sliding_windows, validate=False, kw_args={"window_size": 50, "step_size": 50})),
    ("minmax_scale", FunctionTransformer(minmax_scale, validate=False))
])

In [ ]:
def get_data_with_postprocess(idx, paradigm, dataset, process_pipelines, postprocess_pipeline):
    arr, y, meta = paradigm.get_data(
        dataset=dataset,
        subjects=idx,
        return_raws=False,
        return_epochs=False,
        process_pipelines=process_pipelines,
    )

    arr = postprocess_pipeline.fit_transform(arr)

    assert arr.max() <= 1.000001 and arr.min() >= -1.000001, "Data not scaled to [-1, 1]"
    
    return arr, y, meta

In [ ]:
patient_1 = get_data_with_postprocess([1], paradigm, dataset, process_pipelines, postprocess_pipeline)

print("Array shape:", patient_1[0].shape)
print("y shape:", patient_1[1].shape)
print("Meta keys:", patient_1[2].keys())
print("MAX:", patient_1[0].max())
print("MIN:", patient_1[0].min())
print("within [-1, 1]:", patient_1[0].max() <= 1.000001 and patient_1[0].min() >= -1.000001)

Array shape: (2160, 22, 50)
y shape: (144,)
Meta keys: Index(['subject', 'session', 'run'], dtype='str')
MAX: 0.9999999999999729
MIN: -1.0
within [-1, 1]: True


In [ ]:
train_idx = [1, 2, 3, 4]
test_idx = [5]

In [ ]:
train_data, train_labels, train_meta = get_data_with_postprocess(train_idx, paradigm, dataset, process_pipelines, postprocess_pipeline)
print("Train data shape:", train_data.shape)
print("Train labels shape:", train_labels.shape)
print("Train meta keys:", train_meta.keys())

Train data shape: (8640, 22, 50)
Train labels shape: (576,)
Train meta keys: Index(['subject', 'session', 'run'], dtype='str')


In [ ]:
test_data, test_labels, test_meta = get_data_with_postprocess(test_idx, paradigm, dataset, process_pipelines, postprocess_pipeline)
print("Test data shape:", test_data.shape)
print("Test labels shape:", test_labels.shape)
print("Test meta keys:", test_meta.keys())

Test data shape: (2160, 22, 50)
Test labels shape: (144,)
Test meta keys: Index(['subject', 'session', 'run'], dtype='str')


In [ ]:
import torch.nn.functional as F

class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()

        self.fc = nn.Linear(z_dim, 64 * 4 * 4)

        self.conv1 = nn.Conv2d(64, 32, kernel_size=5, padding=2)

        self.conv2 = nn.Conv2d(32, 16, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm2d(16)

        self.conv3 = nn.Conv2d(16, 8, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm2d(8)

        self.conv4 = nn.Conv2d(8, 1, kernel_size=(1, 15), padding=(0, 0))
        self.bn4 = nn.BatchNorm2d(1)

        # Use stride=(2,1) so spatial height maps 64 -> 22 with kernel (22,1)
        self.conv5 = nn.Conv2d(1, 1, kernel_size=(22, 1), stride=(2, 1), padding=0)

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 64, 4, 4)

        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)  # 8x8
        x = F.relu(self.conv1(x))                                                    # 32x8x8

        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)  # 16x16
        x = F.relu(self.bn2(self.conv2(x)))                                          # 16x16x16

        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)  # 32x32
        x = F.relu(self.bn3(self.conv3(x)))                                          # 8x32x32

        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)  # 64x64
        x = F.relu(self.bn4(self.conv4(x)))                                          # 1x64x50

        x = self.conv5(x)                                                            # 1x22x50
        x = torch.tanh(x)

        return x

In [ ]:
# Quick shape check for generator
G = Generator(z_dim=100)
z = torch.randn(4, 100)
out = G(z)
print("Generator output shape:", out.shape)
print("Expected shape:", (4, 1, 22, 50))
print("Shape match:", tuple(out.shape) == (4, 1, 22, 50))

Generator output shape: torch.Size([4, 1, 22, 50])
Expected shape: (4, 1, 22, 50)
Shape match: True


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 500):
        super().__init__()

        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)

        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, seq_len, d_model)
        seq_len = x.size(1)
        if seq_len > self.pe.size(1):
            raise ValueError(f"seq_len={seq_len} exceeds max_len={self.pe.size(1)}")
        return self.pe[:, :seq_len, :]

In [ ]:
class SAE(nn.Module):
    # num_heads=5 is not specified in the paper might need search, in [1, 2, 5, 10, 25, 50] range based on common practice and paper's mention of "multi-head attention"
    def __init__(self, d_model: int = 50, num_heads: int = 5, ff_dim: int = 100, dropout: float = 0.1, max_len: int = 22):
        super().__init__()

        self.pos_encoding = PositionalEncoding(d_model=d_model, max_len=max_len)
        self.mha = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model),
        )
        
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Supports either (B, 22, 50) or (B, 1, 22, 50).
        if x.dim() == 4 and x.size(1) == 1:
            x = x.squeeze(1)
        if x.dim() != 3:
            raise ValueError(f"Expected 3D input (B, seq_len, d_model), got shape {tuple(x.shape)}")

        x = x + self.pos_encoding(x)
        attn_out, _ = self.mha(x, x, x)
        x = self.norm1(x + self.dropout(attn_out)) # based on equation (7) but in table it list ReLU **might need search**

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x

In [ ]:
# Shape check for SAE block
sae = SAE(d_model=50, num_heads=5, ff_dim=100, dropout=0.1, max_len=22)

x = torch.randn(4, 22, 50)
y = sae(x)
print("Input shape:", tuple(x.shape))
print("Output shape:", tuple(y.shape))
print("Shape match:", tuple(y.shape) == (4, 22, 50))

x4 = torch.randn(4, 1, 22, 50)
y4 = sae(x4)
print("4D input shape:", tuple(x4.shape))
print("4D output shape:", tuple(y4.shape))
print("4D shape match:", tuple(y4.shape) == (4, 22, 50))

Input shape: (4, 22, 50)
Output shape: (4, 22, 50)
Shape match: True
4D input shape: (4, 1, 22, 50)
4D output shape: (4, 22, 50)
4D shape match: True


In [ ]:
class TAE(nn.Module):
    def __init__(self, d_model=22, num_heads=2, ff_dim=44, dropout=0.1, max_len=50):
        super().__init__()
        # paper said TAE ff_dim is different from SAE, but didn't specify the size, so I set it to 2x d_model as a common practice for transformer FFN layers
        # after transpose: (B, 50, 22)
        self.pos_encoding = PositionalEncoding(d_model=d_model, max_len=max_len)

        self.mha = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.dropout1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)

        # TAE is similar to SAE, but with different FC sizes
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model),
        )

        self.dropout2 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Input expected: (B, 22, 50) or (B, 1, 22, 50).
        if x.dim() == 4 and x.size(1) == 1:
            x = x.squeeze(1)  # (B, 22, 50)

        if x.dim() != 3 or x.shape[1:] != (22, 50):
            raise ValueError(f"Expected (B, 22, 50), got {tuple(x.shape)}")

        # Equation (10): transpose -> (B, 50, 22)
        x = x.transpose(1, 2)

        # Equation (11): T1 = T0 + E_pos
        x = x + self.pos_encoding(x)

        # Equation (12): T2 = LN(MultiHead(T1) + T1)
        attn_out, _ = self.mha(x, x, x)
        x = self.norm1(x + self.dropout2(attn_out))

        # Equation (13): T3 = LN(FFN(T2) + T2)
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))

        # transpose back -> (B, 22, 50)
        x = x.transpose(1, 2)
        
        return x

In [ ]:
# Shape check for TAE block
tae = TAE(d_model=22, num_heads=2, ff_dim=44, dropout=0.1, max_len=50)

x = torch.randn(4, 22, 50)
y = tae(x)
print("Input shape:", tuple(x.shape))
print("Output shape:", tuple(y.shape))
print("Shape match:", tuple(y.shape) == (4, 22, 50))

x4 = torch.randn(4, 1, 22, 50)
y4 = tae(x4)
print("4D input shape:", tuple(x4.shape))
print("4D output shape:", tuple(y4.shape))
print("4D shape match:", tuple(y4.shape) == (4, 22, 50))

Input shape: (4, 22, 50)
Output shape: (4, 22, 50)
Shape match: True
4D input shape: (4, 1, 22, 50)
4D output shape: (4, 22, 50)
4D shape match: True


In [ ]:
class Critic(nn.Module):
    def __init__(self, sae: nn.Module, tae: nn.Module):
        super().__init__()

        self.sae = sae
        self.tae = tae

        # Table 3: [B, 44, 50] -> [B, 44, 1]
        self.fc1 = nn.Linear(50, 1)

        # Table 3: then FC -> 1
        self.fc2 = nn.Linear(44, 1)

    def forward(self, x):
        # expected input: (B, 1, 22, 50) or (B, 22, 50)
        if x.dim() == 4 and x.size(1) == 1:
            x = x.squeeze(1)   # (B, 22, 50)

        if x.dim() != 3 or x.shape[1:] != (22, 50):
            raise ValueError(f"Expected (B, 22, 50), got {tuple(x.shape)}")

        # parallel branches
        sae_out = self.sae(x)   # (B, 22, 50)
        tae_out = self.tae(x)   # (B, 22, 50)

        # concat -> (B, 44, 50)
        feat = torch.cat([sae_out, tae_out], dim=1)

        # FC -> (B, 44, 1)
        feat = F.relu(self.fc1(feat))

        # squeeze last dim: (B, 44)
        feat = feat.squeeze(-1)

        # FC -> (B, 1)
        out = self.fc2(feat)

        return out

In [ ]:
# Shape check for Critic block
critic = Critic(sae=sae, tae=tae)

x = torch.randn(4, 22, 50)
y = critic(x)
print("Input shape:", tuple(x.shape))
print("Output shape:", tuple(y.shape))
print("Shape match:", tuple(y.shape) == (4, 1))

x4 = torch.randn(4, 1, 22, 50)
y4 = critic(x4)
print("4D input shape:", tuple(x4.shape))
print("4D output shape:", tuple(y4.shape))
print("4D shape match:", tuple(y4.shape) == (4, 1))

Input shape: (4, 22, 50)
Output shape: (4, 1)
Shape match: True
4D input shape: (4, 1, 22, 50)
4D output shape: (4, 1)
4D shape match: True


In [ ]:
def run_epoch(generator, critic, optimizer_G, optimizer_C, real_data, device):
    batch_size = real_data.size(0)
    z_dim = 100

    # Train Critic
    optimizer_C.zero_grad()

    # Real data
    real_data = real_data.to(device)
    real_validity = critic(real_data)
    real_loss = -real_validity.mean()

    # Fake data
    z = torch.randn(batch_size, z_dim).to(device)
    fake_data = generator(z)
    fake_validity = critic(fake_data.detach())
    fake_loss = fake_validity.mean()

    # Total critic loss
    critic_loss = real_loss + fake_loss
    critic_loss.backward()
    optimizer_C.step()

    # Train Generator
    optimizer_G.zero_grad()

    gen_validity = critic(fake_data)
    gen_loss = -gen_validity.mean()
    gen_loss.backward()
    optimizer_G.step()

    return critic_loss.item(), gen_loss.item()

In [ ]:
def _ensure_channel_dim(x: torch.Tensor) -> torch.Tensor:
    # Keep critic/generator training tensors shape-compatible: (B, 1, 22, 50).
    if x.dim() == 3:
        return x.unsqueeze(1)
    if x.dim() == 4 and x.size(1) == 1:
        return x
    raise ValueError(f"Expected (B,22,50) or (B,1,22,50), got {tuple(x.shape)}")


def compute_gradient_penalty(critic, real_data, fake_data, device):
    real_data = _ensure_channel_dim(real_data).to(device)
    fake_data = _ensure_channel_dim(fake_data).to(device)
    batch_size = real_data.size(0)

    eps_shape = [batch_size] + [1] * (real_data.dim() - 1)
    eps = torch.rand(*eps_shape, device=device)

    x_hat = eps * real_data + (1.0 - eps) * fake_data
    x_hat.requires_grad_(True)

    d_hat = critic(x_hat)

    gradients = torch.autograd.grad(
        outputs=d_hat,
        inputs=x_hat,
        grad_outputs=torch.ones_like(d_hat),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    gradients = gradients.reshape(batch_size, -1)
    gp = ((gradients.norm(2, dim=1) - 1.0) ** 2).mean()

    return gp


def train_critic_step(generator, critic, optimizer_C, real_data, device, z_dim, lambda_gp):
    real_data = _ensure_channel_dim(real_data).to(device)
    batch_size = real_data.size(0)

    optimizer_C.zero_grad()

    z = torch.randn(batch_size, z_dim, device=device)

    # Paper Eq. (1): x_tilde = g(z) + x
    g_z = generator(z)
    x_tilde = g_z + real_data

    d_real = critic(real_data)
    d_fake = critic(x_tilde.detach())

    # Paper Eq. (2): x_hat = eps*x + (1-eps)*x_tilde
    gp = compute_gradient_penalty(critic, real_data, x_tilde.detach(), device)

    # Paper Eq. (3): E[D(fake)] - E[D(real)] + lambda * GP
    wasserstein = d_fake.mean() - d_real.mean()
    critic_loss = d_fake.mean() - d_real.mean() + lambda_gp * gp
    critic_loss.backward()
    optimizer_C.step()

    return {
        "critic_loss": critic_loss.item(),
        "gp": gp.item(),
        "wasserstein": wasserstein.item(),
        "real_score": d_real.mean().item(),
        "fake_score": d_fake.mean().item(),
        "fake_mean": x_tilde.mean().item(),
        "fake_std": x_tilde.std().item(),
    }


def train_generator_step(generator, critic, optimizer_G, real_data, device, z_dim):
    real_data = _ensure_channel_dim(real_data).to(device)
    batch_size = real_data.size(0)

    optimizer_G.zero_grad()

    z = torch.randn(batch_size, z_dim, device=device)

    # Paper Eq. (1): x_tilde = g(z) + x
    g_z = generator(z)
    x_tilde = g_z + real_data

    d_fake = critic(x_tilde)

    # Minimize -E[D(fake)]
    gen_loss = -d_fake.mean()
    gen_loss.backward()
    optimizer_G.step()

    return gen_loss.item()

In [ ]:
def train_batch(generator, critic, optimizer_G, optimizer_C, real_data, n_critic, lambda_gp, z_dim, device):
    critic_metrics = {}

    # Algorithm 1: n = 3 discriminator updates before each generator update
    for _ in range(n_critic):
        critic_metrics = train_critic_step(
            generator, critic, optimizer_C, real_data, device, z_dim, lambda_gp
        )

    gen_loss = train_generator_step(
        generator, critic, optimizer_G, real_data, device, z_dim
    )

    return {
        "critic_loss": critic_metrics["critic_loss"],
        "gen_loss": gen_loss,
        "gp": critic_metrics["gp"],
        "wasserstein": critic_metrics["wasserstein"],
        "real_score": critic_metrics["real_score"],
        "fake_score": critic_metrics["fake_score"],
        "fake_mean": critic_metrics["fake_mean"],
        "fake_std": critic_metrics["fake_std"],
    }

In [ ]:
LAMBDA_GP = 10.0
N_CRITIC = 3
Z_DIM = 100
BATCH_SIZE = 128
LR = 1e-4
NUM_EPOCHS = 100

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


In [ ]:
generator = Generator(z_dim=Z_DIM).to(DEVICE)
sae = SAE(d_model=50, num_heads=5, ff_dim=100, dropout=0.1, max_len=22).to(DEVICE)
tae = TAE(d_model=22, num_heads=2, ff_dim=44, dropout=0.1, max_len=50).to(DEVICE)
critic = Critic(sae=sae, tae=tae).to(DEVICE)

optimizer_G = torch.optim.Adam(generator.parameters(), lr=1e-4, betas=(0.2, 0.999))
optimizer_C = torch.optim.Adam(critic.parameters(), lr=1e-4, betas=(0.2, 0.999))

In [ ]:
# Quick smoke test for one WGAN-GP batch update
real_3d = torch.randn(8, 22, 50, device=DEVICE)
real_4d = torch.randn(8, 1, 22, 50, device=DEVICE)

metrics_3d = train_batch(
    generator, critic, optimizer_G, optimizer_C, real_3d, N_CRITIC, LAMBDA_GP, Z_DIM, DEVICE
)
metrics_4d = train_batch(
    generator, critic, optimizer_G, optimizer_C, real_4d, N_CRITIC, LAMBDA_GP, Z_DIM, DEVICE
)

print("3D batch metrics:", metrics_3d)
print("4D batch metrics:", metrics_4d)

3D batch metrics: {'critic_loss': 6.4954657554626465, 'gen_loss': 0.11275999248027802, 'gp': 0.6478683948516846, 'wasserstein': 0.01678168773651123, 'real_score': -0.11306457221508026, 'fake_score': -0.09628288447856903, 'fake_mean': -0.06601308286190033, 'fake_std': 1.0711673498153687}
4D batch metrics: {'critic_loss': 6.541442394256592, 'gen_loss': 0.03997107967734337, 'gp': 0.6540255546569824, 'wasserstein': 0.0011869296431541443, 'real_score': -0.0509224459528923, 'fake_score': -0.04973551630973816, 'fake_mean': -0.04617229476571083, 'fake_std': 1.0447417497634888}


In [ ]:
X_train = torch.tensor(train_data[:, None, :, :], dtype=torch.float32)
train_loader = torch.utils.data.DataLoader(
    X_train,
    batch_size=128,
    shuffle=True,
    drop_last=True
)   # (N, 1, 22, 50)

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
from pdb import run


history = []

def train(NUM_EPOCHS, run_name, verbose=True):
    with mlflow.start_run(run_name=run_name) as run:
        print(f"Experiment ID: {run.info.experiment_id} Experiment Name: {mlflow.get_experiment(run.info.experiment_id).name}")
        print(f"Run ID: {run.info.run_id} Run Name: {run.info.run_name}")
        print("tracking uri:", mlflow.get_tracking_uri())
        
        mlflow.log_param("lr", LR)
        mlflow.log_param("epochs", NUM_EPOCHS)
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("z_dim", Z_DIM)
        mlflow.log_param("n_critic", N_CRITIC)
        mlflow.log_param("lambda_gp", LAMBDA_GP)
        
        for epoch in tqdm(range(NUM_EPOCHS), desc="Training Epochs"):
            epoch_metrics = {
                "critic_loss": [],
                "gen_loss": [],
                "gp": [],
                "wasserstein": [],
                "real_score": [],
                "fake_score": [],
                "fake_mean": [],
                "fake_std": [],
            }

            for batch in train_loader:
                batch_metrics = train_batch(
                    generator, critic, optimizer_G, optimizer_C, batch, N_CRITIC, LAMBDA_GP, Z_DIM, DEVICE
                )
                for key in epoch_metrics:
                    epoch_metrics[key].append(batch_metrics[key])
            

            # Average metrics over batches
            avg_metrics = {}
            for key, values in epoch_metrics.items():
                avg_metrics[key] = float(np.mean(values)) if len(values) > 0 else float("nan")

            history.append({
                "epoch": epoch + 1,
                **avg_metrics
            })
            
            mlflow.log_metrics(avg_metrics, step=epoch)

            if verbose:
                print(
                    f"Epoch {epoch}/{NUM_EPOCHS} | "
                    f"W: {avg_metrics['wasserstein']:.4f} | "
                    f"C: {avg_metrics['critic_loss']:.4f} | "
                    f"G: {avg_metrics['gen_loss']:.4f} | "
                    f"GP: {avg_metrics['gp']:.4f} | "
                    f"Real: {avg_metrics['real_score']:.4f} | "
                    f"Fake: {avg_metrics['fake_score']:.4f} | "
                    f"Fake μ: {avg_metrics['fake_mean']:.4f} | "
                    f"Fake σ: {avg_metrics['fake_std']:.4f}"
                )
                
    return history

In [ ]:

# mlflow.set_experiment("pytorch_exp")

# with mlflow.start_run():
#     mlflow.log_param("lr", 1e-3)
#     mlflow.log_param("epochs", 20)

#     for epoch in range(20):
#         train_loss = 0.42  # replace with real value
#         val_loss = 0.35
#         val_acc = 0.88

#         mlflow.log_metric("train_loss", train_loss, step=epoch)
#         mlflow.log_metric("val_loss", val_loss, step=epoch)
#         mlflow.log_metric("val_acc", val_acc, step=epoch)

#     torch.save(model.state_dict(), "model.pt")
#     mlflow.log_artifact("model.pt")

In [ ]:
mlflow.set_experiment("stgan_exp_v0")

<Experiment: artifact_location='/Users/ratchanonkhongsawi/Desktop/CMKL/3rd/S2/Research/gans_eeg/gans_eeg/pluem/unet-gans/mlruns/2', creation_time=1774203803014, experiment_id='2', last_update_time=1774203803014, lifecycle_stage='active', name='stgan_exp_v0', tags={}, workspace='default'>

Need check traing sg=heam


In [134]:
RUN_NAME = "Test_run"
history = train(NUM_EPOCHS=NUM_EPOCHS, run_name=RUN_NAME, verbose=True)

Experiment ID: 2 Experiment Name: stgan_exp_v0
Run ID: 89f6ddfc11b140538a8ab8d706f12cfd Run Name: Test_run
tracking uri: sqlite:////Users/ratchanonkhongsawi/Desktop/CMKL/3rd/S2/Research/gans_eeg/gans_eeg/pluem/unet-gans/mlflow.db


Training Epochs:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 0/100 | W: 0.0745 | C: 3.9495 | G: -0.0750 | GP: 0.3875 | Real: 0.0011 | Fake: 0.0757 | Fake μ: -0.0089 | Fake σ: 0.5956
Epoch 1/100 | W: 0.1140 | C: 0.7050 | G: 0.3956 | GP: 0.0591 | Real: -0.4965 | Fake: -0.3824 | Fake μ: -0.0447 | Fake σ: 0.6105
Epoch 2/100 | W: -0.0889 | C: -0.0307 | G: 1.6148 | GP: 0.0058 | Real: -1.5299 | Fake: -1.6188 | Fake μ: 0.0047 | Fake σ: 0.5550
Epoch 3/100 | W: -0.0220 | C: 0.0244 | G: 1.4624 | GP: 0.0046 | Real: -1.4441 | Fake: -1.4660 | Fake μ: 0.0014 | Fake σ: 0.5431
Epoch 4/100 | W: -0.0154 | C: 0.0325 | G: 1.3900 | GP: 0.0048 | Real: -1.3748 | Fake: -1.3902 | Fake μ: 0.0007 | Fake σ: 0.5405
Epoch 5/100 | W: -0.0169 | C: 0.0252 | G: 1.3529 | GP: 0.0042 | Real: -1.3381 | Fake: -1.3549 | Fake μ: -0.0009 | Fake σ: 0.5393
Epoch 6/100 | W: -0.0125 | C: 0.0211 | G: 1.4123 | GP: 0.0034 | Real: -1.4025 | Fake: -1.4150 | Fake μ: -0.0007 | Fake σ: 0.5387
Epoch 7/100 | W: 0.0025 | C: 0.0343 | G: 1.3341 | GP: 0.0032 | Real: -1.3373 | Fake: -1.3348 | Fake μ:

KeyboardInterrupt: 